In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
# 01_mortality_assumption.py
VERSION_ID = "MORT_2026_03"

import pandas as pd
from pyspark.sql.functions import lit

def normalize_hmd_age(age):
    age = str(age)

    # 110+세
    if age.endswith("+"):
        return 100 + int(age.replace("+", ""))

    # 100~109세 (00~09)
    if len(age) == 2 and age.isdigit() and age != "10":
        return 100 + int(age)

    # 0~99세
    return int(age)

# ---------------------------------------------------------
# 1) HMD 파일 읽기 (qx가 이미 있음)
# ---------------------------------------------------------
hmd_path = "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/mortality_HMD.txt"

df_hmd = pd.read_fwf(hmd_path)
df_hmd.columns = ["year", "age", "female", "male", "total"]

df_hmd = df_hmd.melt(
    id_vars=["year", "age"],
    value_vars=["female", "male", "total"],
    var_name="sex",
    value_name="qx_annual"   # 🔥 HMD는 이미 qx 값
)

df_hmd["sex"] = df_hmd["sex"].str.upper().str[0]
df_hmd["age"] = (
    df_hmd["age"].astype(str)
    .apply(normalize_hmd_age)
)
df_hmd["source"] = "HMD"

# deaths_per_100k 없음 → 그대로 qx_annual 사용


# ---------------------------------------------------------
# 2) INSEE 파일 읽기 (100000으로 나눠야 함)
# ---------------------------------------------------------
insee_path = "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/3_Quotients_mortalite.xlsx"

sheets = {
    "FM-Hommes": "M",
    "FM-Femmes": "F"
}

df_list = []

for sheet_name, sex_code in sheets.items():
    df = pd.read_excel(insee_path, sheet_name=sheet_name)
    df = df.rename(columns={df.columns[0]: "year"})

    df_long = df.melt(
        id_vars=["year"],
        var_name="age",
        value_name="deaths_per_100k"
    )

    df_long["age"] = (
        df_long["age"]
        .str.replace(" ans", "")
        .str.replace(" an", "")
        .str.replace("+", "")   # 🔥 "110+" → "110"
        .astype(int)
    )

    # 🔥 INSEE만 qx 계산
    df_long["qx_annual"] = df_long["deaths_per_100k"] / 100000

    df_long["sex"] = sex_code
    df_long["source"] = "INSEE"

    df_list.append(df_long)

df_insee = pd.concat(df_list, ignore_index=True)

# deaths_per_100k 제거
df_insee = df_insee[["year", "age", "sex", "qx_annual", "source"]]


# ---------------------------------------------------------
# 3) 두 파일 합치기
# ---------------------------------------------------------
df_all = pd.concat([df_hmd, df_insee], ignore_index=True)
df_all["version_id"] = VERSION_ID
df_all.loc[
    (df_all["source"] == "INSEE") & (df_all["year"] == 2019),
    "version_id"
] = "MORT_2026_04"

spark_df = spark.createDataFrame(df_all)


# ---------------------------------------------------------
# 4) RAW Delta 테이블 생성
# ---------------------------------------------------------
(
    spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.life_insurance_raw.mortality_assumption")
)


# ---------------------------------------------------------
# 5) 중복 체크
# ---------------------------------------------------------
duplicates = (
    spark_df.groupBy("year", "age", "sex", "source")
            .count()
            .filter("count > 1")
)

display(spark_df)
